# Topic 23a - SQL basics

The main lesson showed one SELECT ... WHERE matching a pandas filter. This notebook covers GROUP BY and JOIN the same way - side by side with the pandas equivalent - because most real data you will gather for the capstone lives in a database, not a CSV someone handed you.

Work through it after reading section "What's new: SQL" of the [main lesson](topic23_capstone_project.ipynb).

In [1]:
# Run this cell as it is.
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def find_data_dir():
    """Locate the lesson's data folder from the repository root, the lesson
    folder, or the assignments folder, so this notebook runs from anywhere."""
    candidates = [
        Path.cwd() / "data",
        Path.cwd() / "capstone_project" / "data",
        Path.cwd().parent / "data",
    ]
    for candidate in candidates:
        if (candidate / "orders.csv").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find orders.csv. Looked in: " + ", ".join(str(c) for c in candidates)
    )


DATA_DIR = find_data_dir()

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 130)
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.color"] = "#e1e0d9"
plt.rcParams["grid.linewidth"] = 0.8
plt.rcParams["axes.edgecolor"] = "#c3c2b7"
plt.rcParams["axes.axisbelow"] = True

PALETTE = {
    "blue": "#2a78d6", "aqua": "#1baf7a", "yellow": "#eda100", "green": "#008300",
    "violet": "#4a3aa7", "red": "#e34948", "magenta": "#e87ba4", "orange": "#eb6834",
}
MUTED = "#c3c2b7"
INK = "#0b0b0b"

print("pandas version:   ", pd.__version__)
print("data folder:      ", DATA_DIR)
print("files:            ", sorted(p.name for p in DATA_DIR.glob("*.csv")))

pandas version:    3.0.6
data folder:       /home/johnny/Projects/itfy-da/capstone_project/data
files:             ['air_quality_readings.csv', 'air_quality_stations.csv', 'customers.csv', 'monthly_targets_wide.csv', 'orders.csv', 'products.csv', 'regions.csv', 'returns.csv']


In [2]:
# The retail "2023 Trading Review" table from Topic 20, reused throughout
# this topic exactly as Topic 22 reused it for the two report types.
def clean_region(series):
    return (series.str.strip().str.replace(r"\s+", " ", regex=True).str.title())


def build_analysis_table(orders, customers, products, regions, returns):
    expected_rows = len(orders)
    customers = customers.copy()
    customers["region"] = clean_region(customers["region"])
    products = products.drop_duplicates(subset="product_id", keep="first")
    customers = customers.merge(regions, on="region", how="left", validate="m:1")
    table = (orders
             .merge(customers, on="customer_id", how="left", validate="m:1")
             .merge(products, on="product_id", how="left", validate="m:1"))
    returned_ids = set(returns["order_id"])
    table["was_returned"] = table["order_id"].isin(returned_ids)
    table["order_date"] = pd.to_datetime(table["order_date"])
    table["month"] = table["order_date"].dt.strftime("%b")
    table["revenue_ghs"] = (table["quantity"] * table["unit_price_ghs"]).round(2)
    table["cost_ghs"] = (table["quantity"] * table["unit_cost_ghs"]).round(2)
    table["profit_ghs"] = (table["revenue_ghs"] - table["cost_ghs"]).round(2)
    if len(table) != expected_rows:
        raise ValueError("Row count changed during wrangling.")
    return table.sort_values("order_date").reset_index(drop=True)


orders = pd.read_csv(DATA_DIR / "orders.csv")
customers = pd.read_csv(DATA_DIR / "customers.csv")
products = pd.read_csv(DATA_DIR / "products.csv")
regions = pd.read_csv(DATA_DIR / "regions.csv")
returns = pd.read_csv(DATA_DIR / "returns.csv")

analysis = build_analysis_table(orders, customers, products, regions, returns)
print(f"Retail analysis table: {analysis.shape[0]} rows x {analysis.shape[1]} columns")

Retail analysis table: 1200 rows x 23 columns


---

## Loading pandas tables into a real database

`sqlite3` ships with Python - no server, no install, no credentials. `DataFrame.to_sql()` loads a table into it in one line, which is also the fastest way to practise SQL against data you already understand.

In [3]:
import sqlite3

conn = sqlite3.connect(":memory:")
orders.to_sql("orders", conn, index=False)
customers.to_sql("customers", conn, index=False)
products.to_sql("products", conn, index=False)

tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'", conn)
print(tables)

        name
0     orders
1  customers
2   products


---

## SELECT, WHERE, ORDER BY

`WHERE` is SQL's boolean filter; `ORDER BY` is SQL's `.sort_values()`.

In [4]:
sql_result = pd.read_sql_query("""
    SELECT order_id, quantity, unit_price_ghs
    FROM orders
    WHERE quantity >= 10
    ORDER BY quantity DESC
    LIMIT 5
""", conn)

pandas_result = (orders[orders["quantity"] >= 10]
                  .sort_values("quantity", ascending=False)
                  [["order_id", "quantity", "unit_price_ghs"]]
                  .head(5))

print("SQL:")
print(sql_result.to_string(index=False))
print("\npandas:")
print(pandas_result.to_string(index=False))

SQL:
order_id  quantity  unit_price_ghs
  O00524        22           20.13
  O00156        22           89.91
  O00371        22            7.98
  O00768        22            8.58
  O00247        21           25.70

pandas:
order_id  quantity  unit_price_ghs
  O00768        22            8.58
  O00371        22            7.98
  O00156        22           89.91
  O00524        22           20.13
  O00247        21           25.70


---

## GROUP BY and aggregates

SQL's `GROUP BY` with `COUNT`/`SUM`/`AVG` is pandas' `.groupby()` with `.size()`/`.sum()`/`.mean()`.

In [5]:
sql_result = pd.read_sql_query("""
    SELECT channel, COUNT(*) AS n_orders, SUM(quantity) AS total_quantity
    FROM orders
    GROUP BY channel
    ORDER BY n_orders DESC
""", conn)

pandas_result = (orders.groupby("channel")
                  .agg(n_orders=("order_id", "count"), total_quantity=("quantity", "sum"))
                  .sort_values("n_orders", ascending=False)
                  .reset_index())

print("SQL:")
print(sql_result.to_string(index=False))
print("\npandas:")
print(pandas_result.to_string(index=False))
print("\nSame numbers:", sql_result["n_orders"].tolist() == pandas_result["n_orders"].tolist())

SQL:
    channel  n_orders  total_quantity
       Shop       543            2382
      Phone       256            1181
     Online       243            1084
Field Agent       158             627

pandas:


    channel  n_orders  total_quantity
       Shop       543            2382
      Phone       256            1181
     Online       243            1084
Field Agent       158             627

Same numbers: True


---

## JOIN

SQL's `JOIN ... ON` is pandas' `.merge(..., on=...)`. The join type keyword matters just as much in SQL as it did in Topic 20 - `LEFT JOIN` keeps every row on the left even where there is no match, exactly like `how="left"`.

In [6]:
sql_result = pd.read_sql_query("""
    SELECT p.category, COUNT(*) AS n_orders
    FROM orders o
    LEFT JOIN products p ON o.product_id = p.product_id
    GROUP BY p.category
    ORDER BY n_orders DESC
""", conn)

pandas_result = (orders.merge(products.drop_duplicates(subset="product_id"), on="product_id", how="left")
                  .groupby("category").size().sort_values(ascending=False)
                  .rename("n_orders").reset_index())

print("SQL:")
print(sql_result.to_string(index=False))
print("\npandas:")
print(pandas_result.to_string(index=False))

SQL:
     category  n_orders
    Beverages       213
       Grains       212
    Household       200
       Snacks       183
Personal Care       168
 Canned Goods       142
 Cooking Oils       103

pandas:


     category  n_orders
    Beverages       213
    Household       200
       Grains       191
       Snacks       183
Personal Care       168
 Canned Goods       142
 Cooking Oils       103


Notice the `products.drop_duplicates(subset="product_id")` in the pandas version - Topic 20's duplicate-key hazard applies here too. The SQL query above did **not** guard against it, and would silently double-count any order against a duplicated product id, exactly the way an unguarded pandas merge would. SQL is not immune to the hazards Topic 20 taught; it just spells them differently (`JOIN` instead of `merge`), and it is just as silent when it goes wrong.

### Try It 1 - a three-way comparison

Write a SQL query joining `orders` to `customers`, grouped by `segment`, returning total `quantity` per segment, sorted descending. Then write the pandas equivalent. Confirm both give the same numbers.

In [7]:
# YOUR CODE HERE

---
## Where this fits

- [Back to the main lesson](topic23_capstone_project.ipynb)
- Previous: [Main lesson](topic23_capstone_project.ipynb)
- Next: [23b - Simple regression](topic23b_simple_regression.ipynb)